In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from accelerate import Accelerator
import os
from PIL import Image
from torchvision import transforms
import numpy as np
from tqdm.auto import tqdm
import random
import matplotlib.pyplot as plt
from torchvision.transforms import functional as TF
from safetensors.torch import load_file
from safetensors.torch import save_file

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [3]:
class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens

class CoordEncoder(nn.Module):
    def __init__(self, embed_dim=32, num_tokens=64):
        super().__init__()
        self.num_tokens = num_tokens
        self.embed_dim = embed_dim

        self.mlp = nn.Sequential(
            nn.Linear(64 * 64, 2048),
            nn.GELU(),
            nn.Linear(2048, num_tokens * embed_dim)  # 64 * 32
        )

    def forward(self, mask):
        mask_ds = F.interpolate(mask, size=(64, 64), mode="nearest")  # (B,1,64,64)
        B = mask_ds.shape[0]
        x = mask_ds.view(B, -1)  # (B, 4096)
        x = self.mlp(x)  # (B, 64*32)
        x = x.view(B, self.num_tokens, self.embed_dim)  # (B,64,32)
        return x


In [ ]:
import torch
import torch.nn.functional as F
from safetensors.torch import load_file
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# =============
# Load Models
# =============

device = "cuda"

# --- load VAE / UNET ---
vae = AutoencoderKL.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="unet").to(device)

# --- latent channels (from SD1.5 = 4) ---
latent_c = vae.config.latent_channels

# --- your trained modules ---
cond_proj = CondEncoder(in_channels=latent_c, out_channels=736).to(device)
coord_encoder = CoordEncoder(embed_dim=32, num_tokens=64).to(device)

# ==========================================================
# Load weights
# ==========================================================

ckpt_dir = "drive/MyDrive/inpaint_best"

unet.load_state_dict(load_file(f"{ckpt_dir}/unet.safetensors"))
cond_proj.load_state_dict(load_file(f"{ckpt_dir}/condencoder.safetensors"))
coord_encoder.load_state_dict(load_file(f"{ckpt_dir}/coordencoder.safetensors"))

vae.eval(); unet.eval(); cond_proj.eval(); coord_encoder.eval()


# ==========================================================
# Utility functions
# ==========================================================

def load_image(path):
    img = Image.open(path).convert("RGB")
    img = img.resize((512, 512))
    img = np.array(img).astype(np.float32) / 255.0
    img = torch.tensor(img).permute(2,0,1)  # (3,H,W)
    img = (img * 2 - 1)  # normalize to [-1,1]
    return img.unsqueeze(0).to(device)  # (1,3,512,512)


def load_mask(path):
    """
    mask：white=mask=1，black=keep=0
    """
    m = Image.open(path).convert("L").resize((512,512))
    m = np.array(m).astype(np.float32) / 255.0
    m = torch.tensor(m).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    return m.to(device)


# ==========================================================
# Inpainting: DDPM reverse sampling
# ==========================================================

def inpaint(image, mask, num_steps=200):

    # 1. Encode image to latent
    with torch.no_grad():
        latent = vae.encode(image).latent_dist.sample() * vae.config.scaling_factor

    B, C, H, W = latent.shape

    # Expand mask to 4 latent channels
    latent_mask = F.interpolate(mask, size=(H,W), mode="nearest")
    latent_mask = latent_mask.expand(-1, C, -1, -1)

    # mask the latent
    masked_latent = latent * (1 - latent_mask)
    m = vae.decode(masked_latent / vae.config.scaling_factor).sample
    m = (m.clamp(-1,1) + 1) / 2
    m = m[0].permute(1,2,0).detach().cpu().numpy()
    plt.figure(figsize=(4,4))
    plt.title("m"); plt.imshow(m); plt.axis("off")
    plt.show()

    # DDPM scheduler
    scheduler = DDPMScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler")
    scheduler.set_timesteps(num_steps, device=device)

    # Initial noise
    noisy = torch.randn_like(latent)

    # ---------------------------------------------
    # DDPM reverse: x_T → x_0
    # ---------------------------------------------
    x = masked_latent + noisy * latent_mask  # start from noise

    for t in scheduler.timesteps:

        # CondEncoder: masked_latents
        cond_tokens = cond_proj(masked_latent)

        # CoordEncoder: mask
        coord_tokens = coord_encoder(mask)

        # merge
        condition = torch.cat([cond_tokens, coord_tokens], dim=-1)  # (B,64,768)

        # Predict noise
        with torch.no_grad():
            noise_pred = unet(x, t, encoder_hidden_states=condition).sample

        # DDPM step
        x = scheduler.step(noise_pred, t, x).prev_sample

        # overwrite known region with original latent
        x = latent_mask * x + (1 - latent_mask) * masked_latent

    # Decode image
    with torch.no_grad():
        image_recon = vae.decode(x / vae.config.scaling_factor).sample

    # convert to display format
    image_recon = (image_recon.clamp(-1,1) + 1) / 2
    return image_recon[0].permute(1,2,0).detach().cpu().numpy()


# ==========================================================
# Example usage
# ==========================================================

image = load_image("region_9.png")
mask  = load_mask("2.png")

result = inpaint(image, mask)

plt.figure(figsize=(14,4))
plt.subplot(1,3,1); plt.title("Input"); plt.imshow((image[0].permute(1,2,0).cpu()+1)/2); plt.axis("off")
plt.subplot(1,3,2); plt.title("Mask"); plt.imshow(mask[0,0].cpu(), cmap='gray'); plt.axis("off")
plt.subplot(1,3,3); plt.title("Inpaint Output"); plt.imshow(result); plt.axis("off")
plt.show()
